In [7]:
# ================================
# Imports (all of them, upfront)
# ================================
import os
import time
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, IA3Config, TaskType, get_peft_model

MODEL_NAME = "xlm-roberta-base"
NUM_LABELS = 3

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

DATA_ROOT = "/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/processed"

def load_base_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS
    ).cuda()

def report_trainable(model, name):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{name} | Trainable: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")


# ================================
# Sanity check — confirm the input is actually attached before doing anything else
# ================================
print(os.listdir(DATA_ROOT))
print(os.listdir(f"{DATA_ROOT}/hi"))


# ================================
# Known gotcha: peft 0.19.1 conflicts with Kaggle's preinstalled torchao 0.10.0
# ================================
!pip uninstall -y torchao


# ================================
# Verify LoRA / DoRA / IA3 all attach correctly
# ================================
lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.1, bias="none",
    task_type=TaskType.SEQ_CLS, target_modules=["query", "value"]
)
lora_model = get_peft_model(load_base_model(), lora_config)
report_trainable(lora_model, "LoRA")

dora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.1, bias="none", use_dora=True,
    task_type=TaskType.SEQ_CLS, target_modules=["query", "value"]
)
dora_model = get_peft_model(load_base_model(), dora_config)
report_trainable(dora_model, "DoRA")

ia3_config = IA3Config(
    task_type=TaskType.SEQ_CLS,
    target_modules=["key", "value", "output.dense"],
    feedforward_modules=["output.dense"]
)
ia3_model = get_peft_model(load_base_model(), ia3_config)
report_trainable(ia3_model, "IA3")


# ================================
# Benchmark: training-step time + peak GPU memory per method
# ================================
def benchmark_training_step(model, batch_size=16, max_length=128, device="cuda"):
    model = model.to(device)
    model.train()

    batch = tokenizer(
        ["This is a test premise."] * batch_size,
        ["This is a test hypothesis."] * batch_size,
        padding=True, truncation=True, max_length=max_length, return_tensors="pt"
    )
    batch = {k: v.to(device) for k, v in batch.items()}
    labels = torch.zeros(batch_size, dtype=torch.long).to(device)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5
    )

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    start = time.perf_counter()
    loss = model(**batch, labels=labels).loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    end = time.perf_counter()

    return {
        "loss": loss.item(),
        "time": end - start,
        "memory": torch.cuda.max_memory_allocated() / 1024**3
    }

fft_result  = benchmark_training_step(load_base_model())
lora_result = benchmark_training_step(get_peft_model(load_base_model(), lora_config))
dora_result = benchmark_training_step(get_peft_model(load_base_model(), dora_config))
ia3_result  = benchmark_training_step(get_peft_model(load_base_model(), ia3_config))

comparison = pd.DataFrame([
    {"Method": "Full Fine-Tuning", "Time (s)": fft_result["time"],  "Peak GPU Memory (GB)": fft_result["memory"]},
    {"Method": "LoRA",             "Time (s)": lora_result["time"], "Peak GPU Memory (GB)": lora_result["memory"]},
    {"Method": "DoRA",             "Time (s)": dora_result["time"], "Peak GPU Memory (GB)": dora_result["memory"]},
    {"Method": "IA3",              "Time (s)": ia3_result["time"],  "Peak GPU Memory (GB)": ia3_result["memory"]},
])
display(comparison)


# ================================
# Tokenizer fragmentation analysis (Hindi vs Telugu)
# ================================
hi = pd.read_parquet(f"{DATA_ROOT}/hi/train_392702.parquet")
te = pd.read_parquet(f"{DATA_ROOT}/te/train_392702.parquet")

def tokenizer_statistics(df, language):
    total_sentences = 0
    total_words = 0
    total_tokens = 0
    split3 = 0
    unk_tokens = 0
    sentence_lengths = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=language):
        text = row["premise"] + " " + row["hypothesis"]
        words = text.split()

        total_sentences += 1
        total_words += len(words)

        encoding = tokenizer(text, add_special_tokens=False)
        tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])

        total_tokens += len(tokens)
        sentence_lengths.append(len(tokens))
        unk_tokens += tokens.count(tokenizer.unk_token)

        for word in words:
            if len(tokenizer.tokenize(word)) >= 3:
                split3 += 1

    return {
        "Language": language,
        "Sentences": total_sentences,
        "Average Words/Sentence": total_words / total_sentences,
        "Average Tokens/Sentence": total_tokens / total_sentences,
        "Tokens/Word": total_tokens / total_words,
        "Words Split ≥3": split3 / total_words * 100,
        "UNK Rate": unk_tokens / total_tokens * 100,
        "Average Sequence Length": np.mean(sentence_lengths),
        "Median Sequence Length": np.median(sentence_lengths),
        "Max Sequence Length": np.max(sentence_lengths)
    }

hi_stats = tokenizer_statistics(hi, "Hindi")
te_stats = tokenizer_statistics(te, "Telugu")

results = pd.DataFrame([hi_stats, te_stats])
display(results)

results.to_csv("/kaggle/working/tokenizer_fragmentation.csv", index=False)


# ================================
# Truncation analysis — confirms max_length=128 is the right choice
# ================================
def truncation_analysis(df, language, tokenizer, max_lengths=[64, 128, 256, 512]):
    sequence_lengths = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=language):
        text = row["premise"] + " " + row["hypothesis"]
        length = len(tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])
        sequence_lengths.append(length)

    sequence_lengths = np.array(sequence_lengths)
    results = []

    for max_len in max_lengths:
        truncated = np.sum(sequence_lengths > max_len)
        results.append({
            "Language": language,
            "Max Length": max_len,
            "Mean Length": round(sequence_lengths.mean(), 2),
            "Median Length": int(np.median(sequence_lengths)),
            "95th Percentile": int(np.percentile(sequence_lengths, 95)),
            "99th Percentile": int(np.percentile(sequence_lengths, 99)),
            "Maximum Length": int(sequence_lengths.max()),
            "Samples Truncated": int(truncated),
            "Truncation Rate (%)": round(truncated / len(sequence_lengths) * 100, 2)
        })

    return pd.DataFrame(results)

hi_trunc = truncation_analysis(hi, "Hindi", tokenizer)
te_trunc = truncation_analysis(te, "Telugu", tokenizer)

truncation_results = pd.concat([hi_trunc, te_trunc], ignore_index=True)
display(truncation_results)

truncation_results.to_csv("/kaggle/working/truncation_analysis.csv", index=False)


# ================================
# Dry run — full mini training loop on the 50-sample Hindi subset
# ================================
from datasets import Dataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score

train_df = pd.read_parquet(f"{DATA_ROOT}/hi/train_50.parquet")
valid_df = pd.read_parquet(f"{DATA_ROOT}/hi/valid.parquet")

train_ds = Dataset.from_pandas(train_df).rename_column("label", "labels")
valid_ds = Dataset.from_pandas(valid_df).rename_column("label", "labels")

MAX_LENGTH = 128

def tokenize(batch):
    return tokenizer(
        batch["premise"], batch["hypothesis"],
        truncation=True, padding="max_length", max_length=MAX_LENGTH
    )

train_ds = train_ds.map(tokenize, batched=True)
valid_ds = valid_ds.map(tokenize, batched=True)

keep = ["input_ids", "attention_mask", "labels"]
train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep])
valid_ds = valid_ds.remove_columns([c for c in valid_ds.column_names if c not in keep])

train_ds.set_format("torch")
valid_ds.set_format("torch")

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=32)

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = {k: v.cuda() for k, v in batch.items()}
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    return total_loss

def evaluate(model, loader):
    model.eval()
    predictions, labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            preds = torch.argmax(outputs.logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())
    return accuracy_score(labels, predictions)

model = load_base_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

loss = train_one_epoch(model, train_loader, optimizer)
acc = evaluate(model, valid_loader)

print("Dry run loss:", loss)
print("Dry run accuracy:", acc)

['manifest.json', 'te', 'hi']
['train_392702.parquet', 'train_500.parquet', 'train_1000.parquet', 'test.parquet', 'train_50.parquet', 'valid.parquet', 'train_100.parquet']
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


LoRA | Trainable: 887,811 / 278,933,766 (0.3183%)


...

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Dry run loss: 4.334489703178406
Dry run accuracy: 0.3333333333333333


In [8]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    print(root)

/kaggle/input
/kaggle/input/notebooks
/kaggle/input/notebooks/venkatkolluu
/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2
/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data
/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/processed
/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/processed/te
/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/processed/hi
/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/raw
/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/raw/.cache
/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/raw/.cache/huggingface
/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/raw/.cache/huggingface/download
/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/raw/.cache/huggingface/download/forward
/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/raw/.cache/huggingface/download/forward/test
/kaggle/input